# Stage 2 — Deduplication + split construction

Goal: Identify problematic samples (outliers, duplicates and cross labels) and tag them in the dataset. Based on this create a clean train + validation set by id (not by image!)

Output: A clean list of unique "sightings."

**! This stage is critical and must come before any training.**

Using frozen embeddings (from hybrid model after initial tests):
* Cluster near-duplicates (camera bursts)
* Assign duplicate_group_id

Create splits:
* train / val / test by group not by image (75/20/5)
* Split Logic: 
    * Train Set: All IDs with $> 5$ images (The "Teachers"). 
    * Query/Gallery (Test) Set: All IDs with $< 5$ images (The "Unknowns"). 
    --> Never mix the IDs. If "Jaguar_01" is in Train, he must never appear in Test.  This is the only way to test if your embeddings actually learned "Jaguar-ness" rather than just memorizing your specific cats.

Decide:
* which IDs are training IDs
* which are eval-only (≤3 images)

Output
* Clean dataset
* Trustworthy splits
* Burst-safe evaluation

**If you skip or weaken this stage → everything later lies.**

Google AI: 
Burst Detection: You can use raw images (background helps).
Data Splitting / Re-ID Prep: You MUST use the blurred background images. If you don't, your model will cheat by memorizing locations instead of learning identities.

# Preparation

## Initial Setup

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "51"

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas/"

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
# Install dependencies
%%capture
%pip install --no-cache-dir -r requirements.txt

In [ ]:
import numpy as np
import faiss
from collections import defaultdict

import fiftyone as fo
from sklearn.metrics import silhouette_score
from collections import deque
import uuid

In [ ]:
from config import NUM_WORKERS, DEVICE, SEED
from utils import ensure_dir
from FiftyOne import FODataset

In [ ]:
DATASET_NAME = "jaguar_stage2_cleaningData"  
BACKBONE = "Hybrid_Dino_MegaDescriptor"
FODATASET_DIR_NAME = "jaguar_export"
PREV_STAGE = "jaguar_stage1.5_deduplication_metadata"
CURRENT_STAGE = f"{DATASET_NAME}_metadata"

In [ ]:
# set paths
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas")
WORK_ROOT = Path("/content")

DATA_ROOT = DRIVE_ROOT / "datasets"
IMG_DIR = DATA_ROOT / FODATASET_DIR_NAME

EMBS_DIR = IMG_DIR / "embeddings"

OUT_DIR = WORK_ROOT / "out"

ensure_dir([OUT_DIR, EMBS_DIR])

## Load FiftyOne Dataset from export folder

In [ ]:
fo_dataset = FODataset.load_from_export(
    export_dir=IMG_DIR,
    dataset_name=DATASET_NAME,
    stage_dir=DATA_ROOT / PREV_STAGE
)
fo_ds = fo_dataset.get_dataset()

session = fo_dataset.launch()

In [ ]:
# Sanity check
print(fo_ds, fo_ds.count())
print(fo_ds.get_field_schema().keys())
print(session.url)

In [ ]:
ds_filtered_by_duplicates = fo_ds.match_tags("duplicate", bool=False)

print(f"Original samples: {len(fo_ds)} | Clean samples: {len(ds_filtered_by_duplicates)}")

fps_filtered = set(ds_filtered_by_duplicates.values("filepath"))
labels_filtered = ds_filtered_by_duplicates.values("ground_truth.label")

# Evaluate Embeddings with Rank 1

In [ ]:
def find_nearest_neighbors(embeddings, k=100):           # neighbors (incl self)
    """
    Performs FAISS Inner Product search on L2-normalized embeddings.
    Returns:
        sims: (N, k) similarity scores (0.0 to 1.0)
        idxs: (N, k) indices of neighbors
    """
    # embs: (N, D) float32, L2-normalized
    embs = embeddings.astype("float32")
    # L2 Normalization makes Inner Product equivalent to Cosine Similarity
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs /= (norms + 1e-12)
    
    D = embs.shape[1]
    index = faiss.IndexFlatIP(D)        # inner product
    index.add(embs)                     # add N vectors
    
    sims, idxs = index.search(embs, k)  # sims: (N,k), idxs: (N,k)
    return sims, idxs

In [ ]:
def evaluate_embedding_quality(embeddings, labels):
    """
    Calculates metrics to rank different embedding models.
    """
    # Check if we have enough classes for Silhouette
    unique_labs = set(labels)
    
    # Only calculate Silhouette if there are at least 2 classes
    if len(unique_labs) > 1:
        # Silhouette Score: Measures cluster "tightness" vs "separation"
        # Range: -1 (worst) to +1 (perfect). Higher is better.
        sil = silhouette_score(embeddings, labels, metric="cosine")
    else:
        sil = 0.0
        print("⚠️ Warning: Only one class found. Silhouette score set to 0.0")
    
    # Rank-1 Accuracy
    _, idxs = find_nearest_neighbors(embeddings, k=2)
    
    hits = 0
    for i in range(len(labels)):
        nearest_neighbor_idx = idxs[i, 1]
        if labels[i] == labels[nearest_neighbor_idx]:
            hits += 1
    
    rank1 = hits / len(labels)
    
    return {"silhouette": sil, "rank1": rank1}

In [ ]:
# duplication from notebook stage 01
# Load specific embeddings
def load_embeddings(model_name, manipulation_mode, save_dir):
    """Loads specific embeddings and their corresponding identifiers."""
    save_dir = Path(save_dir)
    
    emb_file = save_dir / f"embeddings_{model_name}_{manipulation_mode}.npy"
    if not emb_file.exists():
        print(f"❌ Error: Could not find {emb_file}")
        return None
    
    # These usually stay the same across runs if the dataset didn't change          !!TODO: check than what happens with different manipulation modes
    ids_file = save_dir / "sample_ids.npy"
    fps_file = save_dir / "sample_filepaths.npy" 

    # Load the files
    # allow_pickle=True is necessary because sample_ids and filepaths are stored as Python 'object' arrays (strings/IDs)
    embs = np.load(emb_file)
    ids = np.load(ids_file, allow_pickle=True)
    rel_filepaths = np.load(fps_file, allow_pickle=True)
    
    print(f"✅ Loaded {model_name} ({manipulation_mode}): {embs.shape}")

    return embs, ids, rel_filepaths

In [ ]:
manipulation_mode = "original"

model_names = ["DINOv2_Base", "MegaDescriptor_L", "Hybrid_Dino_MegaDescriptor"]

for name in model_names:
    all_embs, all_ids, rel_filepaths = load_embeddings(name, manipulation_mode, EMBS_DIR)

    # Converts relative filepaths to absolute ones
    absolute_filepaths = [str(IMG_DIR / fp) for fp in rel_filepaths]

    # Filter embeddings and paths by the cleaned samples (no duplicates)
    indices = [i for i, fp in enumerate(absolute_filepaths) if str(fp) in fps_filtered]
    embs_filtered = all_embs[indices]

    metrics = evaluate_embedding_quality(embs_filtered, labels_filtered)
    print(f"Model: {name} | Rank-1: {metrics['rank1']:.3f} | Sil: {metrics['silhouette']:.3f}")

## Get Embeddings

In [ ]:
manipulation_mode = "original"

embs, ids, rel_filepaths = load_embeddings(BACKBONE, manipulation_mode, EMBS_DIR)

# Converts relative filepaths to absolute ones
absolute_filepaths = [str(IMG_DIR / fp) for fp in rel_filepaths]

# Filter embeddings and paths by the cleaned samples (no duplicates)
indices = [i for i, fp in enumerate(absolute_filepaths) if str(fp) in fps_filtered]
embs_filtered = all_embs[indices]

# Identify problematic samples: outliers, duplicates and cross labels

A "clean" set for Re-ID requires removing three specific types of noise that frozen embeddings are great at finding:

1) The Burst Effect (Redundancy)
   * **Problem:** If you have 50 photos of a jaguar from the same 2-second burst, and you split them 40/10 between Train/Val, your validation score will be 99% because the model is just recognizing the exact lighting and pose.
   * **Action:** Find clusters with similarity > 0.98. Keep only one representative (the "centroid" of that burst). This forces the model to learn features that generalize across different times/angles.

--> keeping the background here might even be favorable.

2) The "Hard Negative" Mislabels:
    * **Problem:** If Jaguar A and Jaguar B are in a cluster together with 0.95 similarity, either: They are mislabeled (they are actually the same animal). The model is failing to see the unique spots and is only seeing the background. 
    * **Action:** Flag **"Cross-Label Clusters."** If you use MegaDescriptor and it puts two different IDs in the same tight cluster, and DINOv2 does the same, you likely have a labeling error in your source data.
    
3) The "Lonely" Outliers:
    * **Problem:** Images with very low similarity to any other image of the same ID.#
    * **Action:** These are usually bad crops, images of the jaguar's tail only, or extremely blurry shots. Delete these from the training set. They act as "noise" that prevents the embedding space from becoming compact.

## 2) Cross Label
-> Clusters (so similar images) that contain different labels

Action: Tag as hard Negatives for Stage 2.

In [ ]:
def get_connected_clusters(sims, idxs, threshold=0.95):
    """
    Builds a graph where edges exist if similarity >= threshold,
    then finds connected components.
    """
    N = sims.shape[0]
    adj = [[] for _ in range(N)]

    # Build adjacency list only for similarities above threshold
    # Start from index 1 to skip 'self'
    for i in range(N):
        for j, sim in zip(idxs[i, 1:], sims[i, 1:]):
            if sim >= threshold:
                adj[i].append(int(j))
                adj[int(j)].append(i)

    # Connected components using BFS
    comp_id = -np.ones(N, dtype=int)
    clusters = []
    cid = 0

    for i in range(N):
        if comp_id[i] != -1:
            continue

        q = deque([i])
        comp_id[i] = cid
        nodes = [i]

        while q:
            u = q.popleft()
            for v in adj[u]:
                if comp_id[v] == -1:
                    comp_id[v] = cid
                    q.append(v)
                    nodes.append(v)

        if len(nodes) >= 2:  # Only keep non-trivial clusters (size>=2)
            clusters.append(nodes)
        cid += 1

    return clusters

In [ ]:
# Hyperparameters:
k = 50
similarity_threshold = 0.95     # 0.90 finds 676 bursts/duplicates.  |. 0.87 finds 735 bursts/duplicates. | sweet spot: 0.856 finds 761 duplicates and 170 potential keeps

# Find clusters
sims, idxs = find_nearest_neighbors(embs_filtered, k=k)
clusters = get_connected_clusters(sims, idxs, threshold=similarity_threshold)

# Lookup: Map indices back to filenames          
clusters_as_filepaths = []
for cluster in clusters:
    group = [fps_filtered[i] for i in cluster]
    clusters_as_filepaths.append(group)

print(f"Found {len(clusters_as_filepaths)} groups of duplicates.")

In [ ]:
def find_cross_label_clusters(clusters, labels):
    """
    Identifies clusters that contain multiple different labels.
    Returns: List of (indices, unique_labels, counts)
    """
    cross_label_results = []
    for cluster in clusters:
        cluster_labels = [labels[i] for i in cluster]
        unique_labels = sorted(set(cluster_labels))
        
        if len(unique_labels) > 1:
            counts = {l: cluster_labels.count(l) for l in unique_labels}
            cross_label_results.append((cluster, unique_labels, counts))
            
    return cross_label_results

In [ ]:
# Identifies clusters that contain multiple different labels.
cross_labels = find_cross_label_clusters(clusters, labels_filtered)

In [ ]:
# tag as hard negatives in FiftyOne
def tag_cross_labels_in_fiftyone(dataset, cross_labels, filepaths):

    for cluster, _, _ in cross_labels:
        
        cluster_id = f"issue_{uuid.uuid4().hex[:6]}"
        paths = [str(filepaths[i]) for i in cluster]
        
        view = dataset.select_by("filepath", paths)

        view.tag("is_hard_negative")
        view.set_values("quality_issue", ["cross_label"] * len(view))
        view.set_values("issue_cluster_id", [cluster_id] * len(view))

    print(f"Tagged {len(cross_labels)} clusters as 'is_hard_negative'")

In [ ]:
tag_cross_labels_in_fiftyone(fo_ds, cross_labels, fps_filtered)

# show in FiftyOne
view = fo_ds.match_tags("is_hard_negative")
session.view
print(session.url)

## 3) Outliers
-> Same label but low similarity with the rest

Action: Either hard and keep or "delete"

How:
1) Calculate "Self-Similarity": The average similarity of Image A to all other images of the same Jaguar.
2) Calculate sharpness of all samples
3) Rule: If Self-Similarity < 0.5 AND Sharpness is in the bottom 10% of the dataset -> don't use - noise; the rest is hard and to keep (hard_sample)

In [ ]:
def find_low_similarity_outliers(labels, neighbor_sims, z_threshold=2.5):
    """
    Finds samples that are unusually 'lonely' (low similarity to their closest neighbor)
    compared to the average for their specific class.
    """
    best_sims = neighbor_sims.max(axis=1)
    by_label = defaultdict(list)
    for i, lab in enumerate(labels):
        by_label[lab].append(best_sims[i])

    stats = {lab: (np.mean(v), np.std(v)) for lab, v in by_label.items()}
    outliers = []

    for i, lab in enumerate(labels):
        mu, sd = stats[lab]
        if sd < 1e-6: continue # Avoid division by zero for single-sample classes
        
        z = (best_sims[i] - mu) / sd
        if z < -z_threshold:
            outliers.append({
                "index": i, 
                "label": lab, 
                "best_sim": float(best_sims[i]), 
                "z_score": float(z)
            })
    return outliers

In [ ]:
# Images with very low similarity to any other image of the same ID.
outliers = find_low_similarity_outliers(labels_filtered, sims[:, 1:])

In [ ]:
def tag_outliers_in_fiftyone(dataset, outliers, filepaths):
    """
    Tags the dataset based on the analysis.
    """
    # 2. Tag Outliers (Potential Bad Crops or Out-of-Distribution)
    outlier_paths = [filepaths[o["index"]] for o in outliers]
    outlier_view = dataset.select_by("rel_filepath", outlier_paths)
    outlier_view.tag_samples("outlier")
    
    # Optional: store the Z-score so you can sort by it in the App
    for o in outliers:
        sample = dataset[filepaths[o["index"]]] # This works if rel_filepath is the key
        sample["quality_z_score"] = o["z_score"]
        sample.save()
    
    print(f"Tagged {len(outliers)} samples as 'outlier'")

In [ ]:
tag_outliers_in_fiftyone(fo_ds, outliers, list(fps_filtered))

# show in FiftyOne
view = fo_ds.match_tags("outlier")
session.view
print(session.url)

# Save Findings

In [ ]:
fo_dataset.export_metadata_snapshot(DATA_ROOT / CURRENT_STAGE)

## Stop

In [ ]:
# Better (more stable): use average of top-m neighbors for setting uniqueness in fiftyone
## !! TODO: What is now the difference to the outliers??
m = 5
mean_sim = sims[:, :m].mean(axis=1)
uniqueness = 1.0 - mean_sim

if "uniqueness" not in fo_ds.get_field_schema():
    fo_ds.add_sample_field("uniqueness", fo.FloatField)

id2u = {sid: float(u) for sid, u in zip(ids, uniqueness)}

# dataset.set_values("uniqueness", values)
for s in fo_ds.iter_samples(progress=True):
    sid = str(s.id)
    if sid in id2u:
        s["uniqueness"] = id2u[sid]
        s.save()


## !! TODO dazu zuerst dinoembeddings laden und als feld hinzufügen

import fiftyone.brain as fob

# Uniqueness: outlierness in embedding space (needs embeddings)
fob.compute_uniqueness(
    fo_ds,
    embeddings='dino_emb',
)

# Representativeness: how typical a sample is within the embedding space (density-based)
fob.compute_representativeness(
    fo_ds,
    embeddings='dino_emb'
)

# Create Train/Val/Test Splits

1) Filter out bursts: dataset.match(F("is_duplicate") != True)
2) 

In [ ]:
# Only saves new fields and data
#fo_dataset.export_metadata_snapshot(
#    base_export_dir= DATA_ROOT / "stage0_sam3_export",
#)